# DMOSpeech 2: Comprehensive Evaluation & Robustness Benchmark Suite
## Addressing Critical Empirical Questions for Chapter 3

**Author:** Rhenel Jhon Sajol (Master's Thesis 2026)  
**Target Hardware:** NVIDIA CUDA GPU (Google Colab T4 / A100)  
**Scope:** 6 Rigorous Empirical Additions:
1. **WER Discrepancy & Outlier Diagnostic:** Audits CPU vs. GPU Step-8 WER (raw vs. clipped @ 1.0, ASR hallucination loops).
2. **Random Step Baseline:** Tests whether trajectory drift $d_i$ outperforms random candidate step selection matching the exact NFE budget.
3. **SNR-Only LTT Calibration:** Formally calibrates the SNR safety gate $\theta$ via Learn-Then-Test on Calibration split ($N=200$) and evaluates on Held-Out Test split ($N=200$).
4. **TOST Equivalence Testing:** Two One-Sided Tests proving positive quality equivalence within preregistered margins (WER $\pm 0.02$, SIM $\pm 0.02$, UTMOS $\pm 0.10$).
5. **Seed-Noise Control (GPU):** Synthesizes Step 8 against Step 8 with an alternative seed ($s=1337$) to measure the empirical metric noise floor.
6. **DEMAND Babble SNR Sweep (GPU):** Downloads real cafeteria noise (PCAFETER) and tests SNR levels (10, 20, 25, 30 dB) to verify the energy-percentile gate generalizes to real acoustic environments.


### Step 1: Verify GPU Allocation & Python Environment

In [ ]:
!nvidia-smi

### Step 2: Setup Repository & Install Full Dependencies

In [ ]:
import os, sys, subprocess

if not os.path.exists('/content/Shonget-DMOSpeech2') and not os.path.exists('Handover Scheduling'):
    print('[Setup] Cloning repository into /content/Shonget-DMOSpeech2...')
    !git clone https://github.com/Shonget-Thesis/Shonget-DMOSpeech2.git /content/Shonget-DMOSpeech2
    %cd /content/Shonget-DMOSpeech2
    os.chdir('/content/Shonget-DMOSpeech2')
elif os.path.exists('/content/Shonget-DMOSpeech2'):
    %cd /content/Shonget-DMOSpeech2
    os.chdir('/content/Shonget-DMOSpeech2')

print('[Setup] Installing full dependencies...')
%pip install -q torchdiffeq x-transformers ema-pytorch vocos speechbrain transformers jiwer soundfile librosa pyarrow scipy jieba accelerate huggingface_hub zhon

print('[Setup] Environment ready!')

### Step 3: Fast Download Checkpoints & Seed-TTS Audio

In [ ]:
import os
!mkdir -p ckpts
if not os.path.exists('ckpts/model_85000.pt'):
    print('[Download] Downloading speech model checkpoint model_85000.pt...')
    !wget -c -q https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_85000.pt -O ckpts/model_85000.pt

if not os.path.exists('ckpts/model_1500.pt'):
    print('[Download] Downloading duration predictor checkpoint model_1500.pt...')
    !wget -c -q https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_1500.pt -O ckpts/model_1500.pt

test_file = 'data/seed-tts-eval/en/prompt-wavs/common_voice_en_10933823.wav'
if not os.path.exists(test_file) or os.path.getsize(test_file) < 1000:
    print('[Download] Downloading Seed-TTS evaluation dataset audio binaries via huggingface_hub...')
    !rm -rf data/seed-tts-eval
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='zhaochenyang20/seed-tts-eval', repo_type='dataset', local_dir='data/seed-tts-eval')
print('[Download] Models and evaluation audio verified!')

### Step 4: Run Fast Analytical Audits (WER Discrepancy, Random Baseline, SNR-Only LTT, TOST)

In [ ]:
# Runs all 4 analytical modules locally on the verified CPU & GPU datasets
!python "Handover Scheduling/colab/run_comprehensive_evaluation_suite.py" --test wer_discrepancy
!python "Handover Scheduling/colab/run_comprehensive_evaluation_suite.py" --test random_baseline
!python "Handover Scheduling/colab/run_comprehensive_evaluation_suite.py" --test snr_only_ltt
!python "Handover Scheduling/colab/run_comprehensive_evaluation_suite.py" --test tost_equivalence

### Step 5: GPU Benchmark 1 — Seed-Noise Control (Step 8 vs. Step 8)
Synthesizes Step 8 with an alternative seed ($s=1337$) on all 200 held-out test speakers (400 cases: clean & noisy15) and evaluates against the Step 8 reference ($s=42$) using Definition 3.1.  
**Expected Runtime:** ~2 minutes on Tesla T4 GPU.

In [ ]:
# Run seed noise control on CUDA GPU (pass --n_speakers 20 for a quick 20-speaker smoke test, or omit to run all 200)
!python "Handover Scheduling/colab/run_comprehensive_evaluation_suite.py" --test seed_control --device cuda

### Step 6: GPU Benchmark 2 — DEMAND Real-Noise Babble SNR Sweep
Downloads DEMAND cafeteria babble (PCAFETER, 16 kHz) from Zenodo and evaluates real-noise tolerance at 10, 20, 25, and 30 dB SNR across the test speakers.  
**Expected Runtime:** ~10-15 minutes on Tesla T4 GPU (for 200 speakers) or ~2-3 minutes (with `--n_speakers 20`).

In [ ]:
# Run DEMAND real-noise babble SNR sweep on CUDA GPU
# Tip: Add --n_speakers 20 for an initial fast check before running all 200 speakers
!python "Handover Scheduling/colab/run_comprehensive_evaluation_suite.py" --test demand_snr_sweep --device cuda

### Step 7: Summary & Artifact Export

In [ ]:
import os, glob, pandas as pd
res_dir = 'Handover Scheduling/colab/results/comprehensive_suite'
print('Generated audit artifacts in', res_dir, ':')
for f in sorted(glob.glob(os.path.join(res_dir, '*'))):
    print(' -', os.path.basename(f), f'({os.path.getsize(f)} bytes)')

# Pack tar.gz bundle for download
!tar -czvf comprehensive_suite_artifacts.tar.gz "Handover Scheduling/colab/results/comprehensive_suite"